# Octahedral Patchy Particles

We write a basic implementation of octahedral patchy particles with colored interactions to encode for the octahedral DNA self-assembly of particles. The particles are composed of two particle types: **central particles** (which are the DNA origami frame, essentially) and **patches** (which are the particles representing the ssDNA strands emitted at the vertices).

In [1]:
%pip install jax-md

Note: you may need to restart the kernel to use updated packages.


In [1]:
import jax.numpy as jnp
from jax import config
import jax
config.update("jax_enable_x64", True) # necessary for 64-bit precision

from jax import jit, random, grad, value_and_grad, remat, jacfwd, vmap, lax
from jax.example_libraries import optimizers
from jax_md import space, smap, energy, minimize, quantity, simulate, partition, rigid_body, util
# from jax_md.colab_tools import renderer
from jax_md import partition

import numpy as np

## Construction of octahedron

Let us construct a `rigid_body.point_union_shape` to represent the octahedral patchy particle, then visualize it. For the purposes of this demo, I will assign four particles on the equatorial ring (arbitrarily defined) with a species A and the remaining two particles on the poles with a species B.

In [187]:
# @jit ## recommended to not jit this
def octahedron_shape(radius=0.5):
  particle_positions = jnp.array([
      [0.0, 0.0, 0.0],      # central particle position
      [radius, 0.0, 0.0],   # these four are the equatorial patches
      [-radius, 0.0, 0.0],
      [0.0, radius, 0.0],
      [0.0, -radius, 0.0],
      [0.0, 0.0, radius],   # these two are the polar patches
      [0.0, 0.0, -radius]
  ], dtype=jnp.float64)

  # the central particle is species=0, the equatorial are species=1, and polar are species=2
  particle_species = jnp.array([0, 1, 1, 1, 1, 1, 1], dtype=jnp.int32)

  # assigning very small patch masses, with fluctuations on the order of 1e-9, as to avoid gradient instabilities associated with saddle points
  patch_masses = jnp.linspace(0.1, 1.0, 6) * 1e-8
  masses = jnp.concatenate((jnp.array([1.0]), patch_masses), axis=0) # central particle has mass 1.0

  # the collective octahedron is stored in this object
  shape = rigid_body.point_union_shape(particle_positions, masses).set(point_species=particle_species)
  return shape

Let us quickly visualize our octahedron, and make sure it looks fine.

In [188]:
RADIUS = 2.5
oct_shape = octahedron_shape(radius=RADIUS)

# arbitrary initial orientation. last two values must be around this for some reason
oct_orientation = rigid_body.Quaternion(jnp.array([[ 0, 0,  6e-01,  8e-01],]))
oct_rigid_body = rigid_body.RigidBody(center=jnp.array([[10.0, 10.0, 10.0]]), orientation=oct_orientation)
oct_position = vmap(rigid_body.transform, (0, None))(oct_rigid_body, oct_shape).reshape(-1, 3)

In [174]:
# # COLAB ONLY
# species = jnp.array([0, 1, 1, 1, 1, 2, 2], dtype=jnp.int32)
# diameters = jnp.where(species == 0, 1.0, 0.2) * (RADIUS * 2)

# renderer.render(20.0,
#                 {
#                     'particle': renderer.Sphere(oct_position, diameter=diameters),
#                 },
#                 resolution=(512, 512))

## Set up the simulation

In [199]:
# particle count and their specified volume density
number_of_particles = 5
volume_density = 0.03

# get box size
get_box_size = lambda phi, N, rad: (N * jnp.pi * 4 * rad**3 / phi / 3.0) ** (1/3)
box_size = get_box_size(volume_density, number_of_particles, RADIUS)
print("Box Size: {:.3f}".format(box_size))

# set up space
disp_fn, shift_fn = space.periodic(box_size)

# simulation parameters
dt = 1e-3         # time step
num_steps = int(3e5)   # integration time steps (2e5 normally)
max_time = num_steps * dt
# kT = lambda t: 1.0 + (0.1 - 1.0) * (t / max_time)          # temperature
kT = lambda t: 1.0
gamma = 5.0       # friction coefficient of the integrator (for Langevin dynamics)
# The mass and friction coefficient need to be in the form of a 'RigidBody' object.
# The rotational friction coefficient is typically three times the translational coefficient
gamma = rigid_body.RigidBody(jnp.array([gamma]), jnp.array([3.0*gamma]))

save_every = int(1e3)  # view every `save_every` frame in the trajectory whilst visualizing

# energy interaction parameters
D0 = 100.0          # well depth of the interaction energy between patches of same species (100.0 gives decent results)
ALPHA = 250.0       # steepness of the soft-sphere potential (100.0 gives decent results)

# random key
key = random.PRNGKey(0)
print("=== Finished setup ===")

Box Size: 22.178
=== Finished setup ===


Now we need to initialize the positions and orientations of our patchy particles and make sure everything is good. Also, I'm going to initially separate out the positions using `soft_sphere` potentials to make sure each particle is not intersecting with another initially.

In [200]:
central_particle_positions = random.uniform(key, (number_of_particles, 3), minval=0.0, maxval=box_size)

# use soft sphere repulsion to separate particles out, so we don't get weird numerical bugs when running our actual simulation.
energy_fn = energy.soft_sphere_pair(disp_fn, sigma=RADIUS*2)
init_fn, apply_fn = minimize.fire_descent(energy_fn, shift_fn)

state = init_fn(central_particle_positions)
apply_fn_jit = jit(lambda i, state: apply_fn(state))
state = lax.fori_loop(0, 5000, apply_fn_jit, state)

# this should be treated as the actual initial position of the particles.
central_particle_positions = state.position

In [201]:
system_orientation = rigid_body.Quaternion(jnp.array([[ 0, 0,  6e-01,  8e-01] for k in range(number_of_particles)]))
system_rigid_body = rigid_body.RigidBody(center=central_particle_positions, orientation=system_orientation)
system_positions = vmap(rigid_body.transform, (0, None))(system_rigid_body, oct_shape).reshape(-1, 3)

In [202]:
species = jnp.array(list(oct_shape.point_species) * number_of_particles).flatten()
diameters = jnp.where(species == 0, 1.0, 0.2) * (RADIUS * 2)

# # COLAB ONLY
# renderer.render(box_size,
#                 {
#                     'particle': renderer.Sphere(system_positions, diameter=diameters)
#                 },
#                 resolution=(512, 512))

So our `system_` objects store the information about the system of `point_union` particles.

## Simulation of assembly

For now, I'll use Morse potentials with a defined depth `D0` for the patchy interactions, and will use the soft-sphere repulsion between the central particles (since that's more realistic in our case, I suppose).

In [204]:
# Define the energy function

# The two types of patches interact with the same interaction strength, D0,
# but are colored, so the patches only interact with other patches of the same type.
# morse_interaction_matrix = jnp.array([
#     [D0, 0.0],
#     [0.0, D0]
# ])
morse_interaction_matrix = jnp.array([
    [D0]
])
morse_eps = jnp.pad(morse_interaction_matrix, pad_width=(1,0)) # center particles don't interact via morse potential

# Interaction matrix for the LJ(WCA) (not soft sphere for test) interaction is only nonzero for the
# central particle-central particle term
lj_eps = jnp.zeros((2, 2))
lj_eps = lj_eps.at[0, 0].set(1.0)

# pair_energy_soft = energy.soft_sphere_pair(disp_fn, species=3, sigma=RADIUS*2.0, epsilon=soft_eps, alpha=ALPHA)

pair_energy_lj = energy.lennard_jones_pair(disp_fn, species=2, sigma=RADIUS*2.0, epsilon=lj_eps, r_cutoff =RADIUS*2.0*2.0**(1/6.0))

pair_energy_morse = energy.morse_pair(disp_fn, species=2, sigma=0.0, epsilon=morse_eps, alpha=5.0, r_cutoff=1.2)

pair_energy_fn = lambda R, **kwargs: (
    pair_energy_lj(R, **kwargs) +
    pair_energy_morse(R, **kwargs)
)

# convert the energy function to a form which acts on 'RigidBody' objects
energy_fn = rigid_body.point_energy(pair_energy_fn, oct_shape)

# Confirm that we can compute the energy of our initial state

eng = energy_fn(system_rigid_body)
print('Energy of the initial state: {}'.format(eng))

Energy of the initial state: -0.459876612688209


In [205]:
# Here, we simulate without Neighbor Lists
init_fn, step_fn = simulate.nvt_nose_hoover(energy_fn, shift_fn, dt, kT(0.0))
step_fn = jit(step_fn)
state = init_fn(key, system_rigid_body, mass=oct_shape.mass())

do_step = jit(lambda state, t: (step_fn(state, kT=kT(t*dt)), state.position))

final_state, trajectory = lax.scan(do_step, state, jnp.arange(num_steps))

jax.block_until_ready(trajectory)

None

In [183]:
species = jnp.array(list(oct_shape.point_species) * number_of_particles).flatten()
diameters = jnp.where(species == 0, 1.0, 0.2) * (RADIUS * 2)

# transform the trajectory from RigidBody objects to just xyz positions
save_every = 1e3
save_every = int(save_every)
# trajectory_positions = (vmap(vmap(rigid_body.transform, (0, None)), (0, None))(trajectory[::save_every], oct_shape)).reshape(-1, number_of_particles*(1 + 6), 3)

# Compute saved indices once, on-device
n_saved = int(num_steps) // save_every
indices = jnp.arange(n_saved) * save_every  # (n_saved,) integer indices

@jit
def process_trajectory(traj):
    saved = jax.tree_util.tree_map(lambda x: x[indices], traj)
    positions = vmap(
        vmap(rigid_body.transform, (0, None)),
        (0, None)
    )(saved, oct_shape)
    return positions.reshape(n_saved, number_of_particles * 7, 3)

site_positions = process_trajectory(trajectory).block_until_ready()


In [98]:
site_positions[0].shape

(70, 3)

In [14]:
# # COLAB ONLY
# renderer.render(box_size,
#                 {
#                     'particle': renderer.Sphere(site_positions, diameter=diameters)
#                 },
#                 resolution=(512, 512))

In [185]:
traj = site_positions  # shape (num_steps/save_every, 70, 3)

n_frames, n_particles, _ = traj.shape

# define particle types
types = []
radii = []

for i in range(number_of_particles):
    types.append("A")
    radii.append(2.5)        # central particle

    types += ["E"] * 2
    radii += [0.5] * 2       # axis 1 patches

    types += ["E"] * 2       # axis 2 patches
    radii += [0.5] * 2

    # types += ["P"] * 2
    types += ["E"] * 2
    radii += [0.5] * 2       # axis 3 patches

types = np.array(types)
radii = np.array(radii)

with open("trajectory.xyz", "w") as f:
    for t in range(n_frames):
        f.write(f"{n_particles}\n")
        f.write("Properties=species:S:1:pos:R:3:radius:R:1\n")

        for i in range(n_particles):
            x, y, z = traj[t, i]
            r = radii[i]
            f.write(f"{types[i]} {x} {y} {z} {r}\n")